# 03. Avaliação e Análise Estatística

Este notebook avalia o melhor modelo gerado pelo treinamento. Ele calcula métricas, plota a matriz de confusão binária, curvas ROC e PR, gera o Grad-CAM e realiza o Teste de McNemar.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score

from src.config import Config
from src.model import SimpleClassifier
from dataset.loaders import test_loader
import wandb

: 

### Carregar o Melhor Modelo

In [ ]:
checkpoint_path = Config.get_latest_checkpoint()
print(f"Carregando modelo de: {checkpoint_path}")

model = SimpleClassifier.load_from_checkpoint(checkpoint_path)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

### Inferência no Conjunto de Teste

In [ ]:
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for x, y in test_loader:
        logits = model(x.to(device))
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)
        
        # Vamos pegar a probabilidade da classe positiva (COVID-19 = classe 1)
        all_probs.extend(probs[:, 1].cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

CLASSES = ["Pneumonia", "COVID-19"]

### Relatório de Classificação e Matriz de Confusão

In [ ]:
print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.title("Matriz de Confusão Binária")
plt.ylabel("Real")
plt.xlabel("Predição")
plt.show()

### Curvas ROC e Precision-Recall

In [ ]:
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'COVID-19 vs Pneumonia (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
plt.xlabel('Taxa de Falso Positivo (1 - Specificity)')
plt.ylabel('Taxa de Verdadeiro Positivo (Sensitivity)')
plt.title('Curva ROC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Teste Estatístico (McNemar)
O Teste de McNemar é usado para comparar as proporções de erros pareados. Se você salvar as previsões deste notebook e de outro experimento (ex: FULL vs SEGMENTED), você pode carregá-las aqui.

In [ ]:
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

# Salvar predições para comparar depois
df_preds = pd.DataFrame({'real': all_labels, 'pred_atual': all_preds})
df_preds.to_csv(f"../outputs/preds_{Config.EXPERIMENT_NAME}.csv", index=False)
print(f"Predições salvas em outputs/preds_{Config.EXPERIMENT_NAME}.csv para futuros testes de McNemar.")